# VM vs Container Performance Analysis

This notebook performs the final Python-based analysis for the **VM vs Docker Container Performance Experiment**.

It uses the existing processed measurements under `results/processed/` and the existing figures under `results/figures/`.

### Analysis covered

- CPU performance
- CPU scalability
- Memory throughput
- Disk bandwidth
- Disk IOPS
- Network throughput
- FastAPI throughput
- FastAPI latency
- Startup time
- API scalability
- Repeated-run statistics
- VM vs Container percentage differences
- Final observations and limitations

> **Important:** This notebook analyzes the measurements already collected. It does **not** rerun any benchmark.

## Experimental configuration and important deviation

The benchmark campaign compared an Ubuntu VM with a Docker container under controlled CPU and memory limits.

The actual VM had approximately **4.8 GiB RAM**, so the Sysbench memory workload was reduced to **2 GiB** instead of the laboratory manual's 10 GiB workload. This limitation is documented in `docs/methodology.md` and `docs/vm-configuration.txt`.

The notebook reports the measured values as collected and does not replace them with example values from the manual.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Image, Markdown

# Locate the project root robustly.
cwd = Path.cwd()
candidates = [cwd, *cwd.parents]

PROJECT_ROOT = None
for p in candidates:
    if (p / "results" / "processed").is_dir():
        PROJECT_ROOT = p
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate project root. Run this notebook from the project "
        "or a subdirectory containing results/processed/."
    )

PROCESSED = PROJECT_ROOT / "results" / "processed"
FIGURES = PROJECT_ROOT / "results" / "figures"

print("Project root:", PROJECT_ROOT)
print("Processed results:", PROCESSED)
print("Figures:", FIGURES)

In [ ]:
# Load all processed datasets used by the analysis.

FILES = {
    "cpu": "cpu_results.csv",
    "cpu_scalability": "cpu_scalability.csv",
    "memory": "memory_results.csv",
    "disk": "disk_results.csv",
    "network": "network_results.csv",
    "application": "application_results.csv",
    "startup": "startup_results.csv",
    "api_scalability": "api_scalability.csv",
    "statistics": "statistics.csv",
    "final_comparison": "final_comparison.csv",
}

dfs = {}
for key, filename in FILES.items():
    path = PROCESSED / filename
    if not path.exists():
        raise FileNotFoundError(f"Missing processed file: {path}")
    dfs[key] = pd.read_csv(path)

for key, df in dfs.items():
    print(f"{key:18s} -> {df.shape[0]:>3} rows x {df.shape[1]:>2} columns")

In [ ]:
# Normalize column names so the notebook works with the CSVs generated by the project scripts.

def rename_aliases(df, aliases):
    rename = {}
    lower = {c.lower().strip(): c for c in df.columns}
    for target, choices in aliases.items():
        if target in df.columns:
            continue
        for choice in choices:
            if choice.lower() in lower:
                rename[lower[choice.lower()]] = target
                break
    return df.rename(columns=rename)

COMMON = {
    "system": ["system", "environment", "env"],
}

dfs["cpu"] = rename_aliases(dfs["cpu"], {
    **COMMON, "events_per_second": ["events_per_second", "events/sec", "events_sec", "eventspersecond"]
})
dfs["cpu_scalability"] = rename_aliases(dfs["cpu_scalability"], {
    **COMMON, "threads": ["threads", "thread"],
    "events_per_second": ["events_per_second", "events/sec", "events_sec", "eventspersecond"]
})
dfs["memory"] = rename_aliases(dfs["memory"], {
    **COMMON, "throughput_mib_per_sec": ["throughput_mib_per_sec", "throughput", "bandwidth", "mib_per_sec", "memory_throughput"]
})
dfs["disk"] = rename_aliases(dfs["disk"], {
    **COMMON, "operation": ["operation", "test", "workload"],
    "bandwidth": ["bandwidth", "bandwidth_mib_s", "throughput_mib_per_sec", "throughput"],
    "iops": ["iops"]
})
dfs["network"] = rename_aliases(dfs["network"], {
    **COMMON, "test": ["test", "operation", "benchmark"],
    "throughput_gbps": ["throughput_gbps", "throughput", "bandwidth_gbps"]
})
dfs["application"] = rename_aliases(dfs["application"], {
    **COMMON, "endpoint": ["endpoint", "test", "workload"],
    "requests_per_second": ["requests_per_second", "req_per_sec", "requests_sec", "throughput"],
    "mean_latency_ms": ["mean_latency_ms", "latency_ms", "mean_latency"]
})
dfs["startup"] = rename_aliases(dfs["startup"], {
    **COMMON, "startup_ms": ["startup_ms", "startup_time_ms", "time_ms", "startup_time"]
})
dfs["api_scalability"] = rename_aliases(dfs["api_scalability"], {
    **COMMON, "workload": ["workload", "level", "threads"],
    "requests_per_second": ["requests_per_second", "req_per_sec", "requests_sec", "throughput"],
    "latency_ms": ["latency_ms", "mean_latency_ms", "latency", "mean_latency"]
})

print("Normalized columns:")
for key in ["cpu", "cpu_scalability", "memory", "disk", "network", "application", "startup", "api_scalability"]:
    print(f"{key:18s}: {list(dfs[key].columns)}")


## 1. CPU performance

The CPU benchmark uses Sysbench with a prime limit of 20,000 and four threads for the repeated-run comparison.

The repeated results are stored in `cpu_results.csv`.

In [ ]:
cpu = dfs["cpu"].copy()

display(cpu)

cpu_summary = (
    cpu.groupby("system", as_index=True)["events_per_second"]
       .agg(["mean", "median", "min", "max", "std"])
       .round(2)
)

print("CPU repeated-run statistics")
display(cpu_summary)

vm_cpu = cpu_summary.loc["vm", "mean"]
container_cpu = cpu_summary.loc["container", "mean"]
cpu_diff = (container_cpu - vm_cpu) / vm_cpu * 100

print(f"Container vs VM CPU difference: {cpu_diff:+.2f}%")

## 2. CPU scalability

CPU scalability was measured at 1, 2, 4, and 8 threads.

In [ ]:
cpu_scalability = dfs["cpu_scalability"].copy()
display(cpu_scalability)

pivot = cpu_scalability.pivot(
    index="threads",
    columns="system",
    values="events_per_second"
).sort_index()

display(pivot.round(2))

scale = pivot.copy()
scale["container_vs_vm_percent"] = (
    (scale["container"] - scale["vm"]) / scale["vm"] * 100
)

display(scale.round(2))

## 3. Memory throughput

The memory results use the measured 2 GiB workload described in the methodology.

In [ ]:
memory = dfs["memory"].copy()

memory_summary = (
    memory.groupby("system")["throughput_mib_per_sec"]
          .agg(["mean", "median", "min", "max", "std"])
          .round(2)
)

display(memory_summary)

vm_mem = memory_summary.loc["vm", "mean"]
container_mem = memory_summary.loc["container", "mean"]
memory_diff = (container_mem - vm_mem) / vm_mem * 100

print(f"Container vs VM memory-throughput difference: {memory_diff:+.2f}%")

## 4. Disk performance

Disk testing covers sequential read, sequential write, random read, and random write.

Both bandwidth and IOPS are retained in the processed dataset.

In [ ]:
disk = dfs["disk"].copy()
display(disk)

disk_bw = disk.pivot(index="operation", columns="system", values="bandwidth")
disk_bw["container_vs_vm_percent"] = (
    (disk_bw["container"] - disk_bw["vm"]) / disk_bw["vm"] * 100
)

print("Disk bandwidth comparison")
display(disk_bw.round(2))

disk_iops = disk.pivot(index="operation", columns="system", values="iops")
disk_iops["container_vs_vm_percent"] = (
    (disk_iops["container"] - disk_iops["vm"]) / disk_iops["vm"] * 100
)

print("Disk IOPS comparison")
display(disk_iops.round(2))

## 5. Network throughput

Network measurements contain the single-stream `iperf3` test and the parallel-stream `iperf3_P4` test.

These measurements represent the network path used in the experiment and should not be interpreted as general Internet throughput.

In [ ]:
network = dfs["network"].copy()
display(network)

network_cmp = network.pivot(index="test", columns="system", values="throughput_gbps")
network_cmp["container_vs_vm_percent"] = (
    (network_cmp["container"] - network_cmp["vm"]) / network_cmp["vm"] * 100
)

display(network_cmp.round(2))

## 6. FastAPI application performance

Two application endpoints were benchmarked:

- `/health` — lightweight application workload
- `/compute` — compute-oriented workload

The processed dataset contains requests per second and mean latency.

In [ ]:
application = dfs["application"].copy()
display(application)

app_req = application.pivot(
    index="endpoint",
    columns="system",
    values="requests_per_second"
)
app_req["container_vs_vm_percent"] = (
    (app_req["container"] - app_req["vm"]) / app_req["vm"] * 100
)

print("Application throughput")
display(app_req.round(2))

app_lat = application.pivot(
    index="endpoint",
    columns="system",
    values="mean_latency_ms"
)
app_lat["container_vs_vm_percent"] = (
    (app_lat["container"] - app_lat["vm"]) / app_lat["vm"] * 100
)

print("Application latency")
display(app_lat.round(2))

## 7. Startup time

Five startup measurements were collected for each environment.

In [ ]:
startup = dfs["startup"].copy()
display(startup)

startup_summary = (
    startup.groupby("system")["startup_ms"]
           .agg(["mean", "median", "min", "max", "std"])
           .round(2)
)

display(startup_summary)

vm_start = startup_summary.loc["vm", "mean"]
container_start = startup_summary.loc["container", "mean"]
startup_diff = (container_start - vm_start) / vm_start * 100

print(f"Container vs VM startup-time difference: {startup_diff:+.2f}%")

## 8. API scalability

The experiment used four increasing API workload levels.

Mapping used for this analysis:

| Workload | Configuration |
|---|---|
| 1 | 1 thread / 10 connections |
| 2 | 2 threads / 50 connections |
| 4 | 4 threads / 100 connections |
| 8 | 4 threads / 200 connections |

In [ ]:
api = dfs["api_scalability"].copy()
display(api)

api_req = api.pivot(index="workload", columns="system", values="requests_per_second").sort_index()
api_req["container_vs_vm_percent"] = (
    (api_req["container"] - api_req["vm"]) / api_req["vm"] * 100
)

print("API throughput scalability")
display(api_req.round(2))

api_lat = api.pivot(index="workload", columns="system", values="latency_ms").sort_index()
api_lat["container_vs_vm_percent"] = (
    (api_lat["container"] - api_lat["vm"]) / api_lat["vm"] * 100
)

print("API latency scalability")
display(api_lat.round(2))

## 9. Statistical summary

The project calculates mean, median, minimum, maximum, and standard deviation for the repeated CPU, memory, and startup measurements.

In [ ]:
statistics = dfs["statistics"].copy()
display(statistics.round(2))

## 10. Final comparison dataset

This dataset contains the consolidated VM-versus-container comparison generated from the processed measurements.

In [ ]:
final_comparison = dfs["final_comparison"].copy()
display(final_comparison)

## 11. Existing result figures

The benchmark campaign generated ten figures. The notebook displays the existing files rather than creating a second inconsistent set of graphs.

In [ ]:
figure_names = [
    "01_cpu_performance.png",
    "02_memory_performance.png",
    "03_disk_bandwidth.png",
    "04_disk_iops.png",
    "05_network_throughput.png",
    "06_application_throughput.png",
    "07_application_latency.png",
    "08_startup_time.png",
    "09_cpu_scalability.png",
    "10_api_scalability.png",
]

for filename in figure_names:
    path = FIGURES / filename
    print(f"\n{filename}")
    if path.exists():
        display(Image(filename=str(path)))
    else:
        print(f"Missing figure: {path}")

## 12. Main observations from the measured data

The following observations are generated from the processed results:

- CPU averages are close between the VM and container environments.
- Memory throughput differs substantially under the selected 2 GiB workload.
- Disk behavior varies by access pattern, so one storage result should not be generalized to every I/O workload.
- Network results are relatively close for the two tested `iperf3` configurations.
- Application behavior differs between the lightweight health endpoint and the compute endpoint.
- Startup time shows a measurable difference between the two environments.
- Increasing API workload changes both throughput and latency.

In [ ]:
# Generate a compact, data-derived observation table for the report.

observations = pd.DataFrame([
    ["CPU mean events/sec", vm_cpu, container_cpu, cpu_diff],
    ["Memory mean MiB/sec", vm_mem, container_mem, memory_diff],
    ["Startup mean ms", vm_start, container_start, startup_diff],
], columns=["Metric", "VM", "Container", "Container_vs_VM_%"])

display(observations.round(2))

## 13. Limitations

1. **Memory workload deviation:** the manual specifies a 10 GiB memory workload, but the actual VM had approximately 4.8 GiB RAM, so the measured workload was reduced to 2 GiB.

2. **Single experimental platform:** the measurements represent the specific hardware, VM configuration, Docker configuration, software versions, and storage used in this study.

3. **Network scope:** the `iperf3` results describe the selected local/virtual path, not Internet performance.

4. **Application scope:** the FastAPI workload is a controlled microbenchmark and is not representative of every production application.

5. **Scalability scope:** the API scalability test covers the implemented workload levels and does not evaluate Kubernetes replicas, autoscaling, or multi-host deployment.

## 14. Reproducibility

This notebook expects the repository structure:

```text
vm-vs-container-performance/
├── results/
│   ├── processed/
│   └── figures/
├── analysis/
│   └── analysis.ipynb
└── ...
```

From the project root, the processed datasets and figures can be regenerated using the project's existing analysis scripts:

```bash
python analysis/process_results.py
python analysis/create_comparison.py
python analysis/generate_plots.py
```

Then open this notebook:

```bash
jupyter notebook
```

No benchmark command is executed by this notebook.

## 15. Final conclusion

The experiment provides a multi-dimensional comparison of VM and container behavior using actual measurements.

The results show that performance is workload-dependent: CPU behavior is close in the tested configuration, while differences are visible in memory, storage, application, startup, and scalability measurements.

The conclusions in the final report should be interpreted together with the documented VM/container configuration, repeated measurements, statistical summaries, and the memory-workload limitation.